# 参数化单轴运动调试
**默认仅预览，不移动任何电机。** 配置为虚构标定例子，不是四探针实际R/X/Z参数。毫米/角度换算、步数与速度量化、脉冲宽度、软硬限位和减速表可清晰调整。


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT/'scripts/motion_tuning.py').is_file(): ROOT = ROOT.parent
assert (ROOT/'scripts/motion_tuning.py').is_file()
sys.path.insert(0, str(ROOT/'scripts'))
from motion_tuning import AxisConfig, preview_move, fingerprint, authorize, execute_plan
adapter = None  # 用户需提供已验证IP/寄存器适配器；此模板不加载Overlay


## 1. 机械标定、时钟及限位
gear_ratio=电机转数/输出转数；直线轴units_per_output_rev是导程mm/rev，旋转轴通常360deg/rev。硬限位必须来自实际PL设计/能力契约，不允许为了通过校验扩大。


In [ ]:
AXIS_NAME = 'example_linear'
UNIT = 'mm'
FULL_STEPS_PER_REV = 200
MICROSTEPS = 16
GEAR_RATIO = 1.0
UNITS_PER_OUTPUT_REV = 2.0
PL_CLOCK_HZ = 50_000_000
PL_HARD_MIN_STEPS = 0
PL_HARD_MAX_STEPS = 16000
SOFT_MIN = 0.0
SOFT_MAX = 10.0
MAX_SPEED = 20.0
MIN_PULSE_CYCLES = 2
axis = AxisConfig(axis=AXIS_NAME, unit=UNIT, full_steps_per_rev=FULL_STEPS_PER_REV, microsteps=MICROSTEPS, gear_ratio=GEAR_RATIO, units_per_output_rev=UNITS_PER_OUTPUT_REV, clock_hz=PL_CLOCK_HZ, hard_min_steps=PL_HARD_MIN_STEPS, hard_max_steps=PL_HARD_MAX_STEPS, soft_min=SOFT_MIN, soft_max=SOFT_MAX, max_speed=MAX_SPEED, min_pulse_cycles=MIN_PULSE_CYCLES)
print('steps per', UNIT, '=', axis.steps_per_unit)


## 2. 调整目标、速度和减速；先预览
周期指完整STEP周期。该模板LUT按周期线性插值，**不是恒加速度**，也不宣称等同于当前四探针PL减速表。真实PL不支持此profile时执行被拒绝。单次距离小于窗口时有效窗口缩小，报告列出请求与有效值。


In [ ]:
OFFLINE_PREVIEW = True
SIMULATED_CURRENT_STEPS = 0
TARGET = 1.0
SPEED = 5.0
END_SPEED = 1.0
DECEL_WINDOW_STEPS = 100
DECEL_SEGMENTS = 8
if OFFLINE_PREVIEW:
    current_steps, homed, trusted = SIMULATED_CURRENT_STEPS, True, True
else:
    if adapter is None: raise RuntimeError('Supply a validated adapter; never guess register addresses')
    state = adapter.status()
    current_steps, homed, trusted = state['position_steps'], state['homed'], state['trusted']
plan = preview_move(axis, current_steps, TARGET, SPEED, END_SPEED, DECEL_WINDOW_STEPS, DECEL_SEGMENTS, homed, trusted)
ARM_MOTION = False  # 每次修改/重做预览撤销旧授权
print(json.dumps(plan, indent=2))
print('preview SHA:', fingerprint(plan))


## 3. 显式人工授权和真实适配器入口（默认禁用）
当前工程X轴无立即软件STOP，不能用这个入口冒充支持；请保持预览。所有adapter方法必须有界/非阻塞。ctrl-c或timeout只会请求stop，不是物理急停保证。步骤计数估计不是编码器实测。


In [ ]:
ARM_MOTION = False  # 人工核验后才改True；重新预览会撤销
MECHANISM_CONFIRMED = False
ESTOP_CONFIRMED = False
LIMITS_CONFIRMED = False
RUN_MOVE = False
if not RUN_MOVE:
    print('PREVIEW ONLY: no motion, no register access.')
else:
    if OFFLINE_PREVIEW or not ARM_MOTION or adapter is None:
        raise PermissionError('Disable offline mode; verify adapter and freshly arm after preview')
    receipt = authorize(plan, mechanism_confirmed=MECHANISM_CONFIRMED, estop_confirmed=ESTOP_CONFIRMED, limits_confirmed=LIMITS_CONFIRMED)
    try:
        final_state = execute_plan(adapter, plan, receipt, timeout_s=5.0, poll_s=0.01)
        print('Adapter result (retain independent physical verification):', final_state)
    finally:
        ARM_MOTION = False
        RUN_MOVE = False


## 4. 保存参数预览，用于对比调试
保存文件只保存配置，不执行运动、不保存虚构实际位移。真实测试另保存actual encoder/测量工具读数和时间。


In [ ]:
SAVE_PREVIEW = False
if SAVE_PREVIEW:
    (ROOT/'examples/motion_preview.json').write_text(json.dumps(dict(plan=plan, plan_sha256=fingerprint(plan)), indent=2)+'\n', encoding='utf-8')
